# Handwired GPU fracture mechanics — native NVIDIA Vulkan
Runs actual GPU cohesive bond forces, irreversible fracture and Handwired-compiled numerical integration against independently written CPU model. Does not certify FEM or browser WebGPU.


In [ ]:
import os,subprocess,time,pathlib,json,traceback
root=pathlib.Path('/kaggle/working')
report={'ok':False,'stages':[],'browserWebGpuVerified':False}
def run(label,command):
 print('##',label,flush=True)
 proc=subprocess.run(command,shell=True,executable='/bin/bash',text=True,
  stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=1400)
 print(proc.stdout[-6000:],flush=True)
 report['stages'].append({'step':label,'exitCode':proc.returncode,
  'outputTail':proc.stdout[-2000:]})
 (root/'fracture-vulkan-job.json').write_text(json.dumps(report,indent=2))
 if proc.returncode!=0:raise RuntimeError(label+' failed')
try:
 run('Real NVIDIA T4 GPU identity','nvidia-smi --query-gpu=name,driver_version --format=csv,noheader')
 run('Install compatible NVIDIA Vulkan ICD','apt-get update -qq && apt-get install -y -qq libnvidia-gl-580 libvulkan1 vulkan-tools')
 run('Verify physical Vulkan device','vulkaninfo --summary | grep -E "deviceName|deviceType|driverName" | head -14')
 run('Install native wgpu and Bun','python -m pip -q install wgpu==0.32.0 && curl -fsSL https://bun.sh/install | bash')
 run('Fetch actual fracture WGSL branch','git clone --depth 1 --branch feat/frontier-06-destruction https://github.com/Chrisbryan17/handwired.git /kaggle/temp/handwired-fracture')
 run('Install pinned Bun dependencies','cd /kaggle/temp/handwired-fracture && /root/.bun/bin/bun install --frozen-lockfile')
 run('Export real Handwired structural WGSL and independent CPU baseline',
  'cd /kaggle/temp/handwired-fracture && /root/.bun/bin/bun scripts/export-fracture-native.ts /kaggle/working')
 run('Native T4 cohesive-bond fracture and GPU vs CPU parity',
  'python /kaggle/temp/handwired-fracture/ops/kaggle/native-fracture/verify_fracture.py /kaggle/working')
 report['evidence']=json.loads((root/'native-fracture-report.json').read_text())
 report['ok']=report['evidence'].get('ok',False)
except Exception as exc:
 report['error']=str(exc);traceback.print_exc()
finally:
 report['finished']=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())
 (root/'fracture-vulkan-job.json').write_text(json.dumps(report,indent=2))
print('FINAL_NATIVE_FRACTURE',report['ok'],flush=True)
